# Cloud Workload Forecasting: Exploratory Data Analysis (EDA)

**Goal:** understand the data *before* building any model.
Each section has: **Question → What I did → What I found → Decision**.

**Rule:** the last 24 timestamps (2 hours) are the *holdout* I must forecast later.
Everything about time below uses **only the first 110 timestamps (training window)**.
(An early draft printed the holdout mean once. I removed it. I chose my model using only the training window and scored the holdout once at the end.)

**Libraries used:** pandas, numpy, matplotlib, seaborn.

| Word | Simple meaning |
|---|---|
| **VM** | a virtual computer in the cloud |
| **Training window** | first 110 time points: allowed to learn from |
| **Holdout** | last 24 time points: must predict, never learn from |
| **Persistence** | simplest forecast: "future = last value" |
| **Mean reversion** | values drift back to their average |
| **Autocorrelation** | how much a value looks like earlier values |
| **Leakage** | accidentally using future information |

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
os.makedirs("../reports/figures", exist_ok=True)      # folder for saved pictures

# 1) load the data with pandas
cpu = pd.read_csv("../challenge_data/vm_cpu_readings.csv.gz")
meta = pd.read_csv("../challenge_data/vm_metadata.csv.gz",
                   dtype={"virtual_core_bucket": str, "memory_gb_bucket": str})   # buckets like ">24" are text

# 2) split by time: last 24 timestamps = holdout, the rest = training
all_times = np.sort(cpu["timestamp"].unique())
train_times = all_times[:-24]

# 3) keep only training rows and add the VM labels (category, cores, memory)
tr = cpu[cpu["timestamp"].isin(train_times)]
tr = tr.merge(meta[["vm_id", "vm_category", "virtual_core_bucket", "memory_gb_bucket"]], on="vm_id")

# 4) time in hours (easier to read on plots)
hours = (train_times - train_times[0]) / 3600

print("training timestamps:", len(train_times))
print("last training timestamp:", train_times[-1])      # expect 32700

ModuleNotFoundError: No module named 'seaborn'

## 1. What is in the files?
**Question:** What do the two files look like?

In [ ]:
print("CPU readings:", cpu.shape)
print("VM metadata :", meta.shape)
print(cpu.head(3))
print(meta.head(3))

**Found:** the CPU file has one row per VM per 5 minutes (`min_cpu`, `max_cpu`, `avg_cpu`). The metadata file has one row per VM.
**Decision:** two targets per timestamp: the number of VMs (`active_vm_count`) and the average CPU over VMs (`aggregate_cpu_utilization`).

## 2. Is the data clean?
**Question:** Any missing values, duplicates, or gaps in time?

In [ ]:
gaps = np.diff(all_times)

print("missing values       :", cpu.isna().sum().sum())
print("duplicate rows       :", cpu.duplicated(["vm_id", "timestamp"]).sum())
print("time steps (seconds) :", set(gaps))                          # expect only {300}
print("readings per VM      :", set(cpu.groupby("vm_id").size()))   # expect only {134}

**Found:** nothing missing, no duplicates, every step is 300 seconds (5 minutes), every VM has all 134 readings.
**Decision:** no cleaning or filling needed.

## 3. Do VMs come and go?
**Question:** Does the number of active VMs change?

In [ ]:
vm_count = tr.groupby("timestamp")["vm_id"].nunique()      # distinct VMs per timestamp

plt.figure(figsize=(7, 3))
sns.lineplot(x=hours, y=vm_count.values)
plt.ylim(0, vm_count.max() * 1.2)
plt.title("Number of active VMs")
plt.xlabel("hours"); plt.ylabel("VMs")
plt.savefig("../reports/figures/03_vm_count.png", dpi=150, bbox_inches="tight")
plt.show()

print("VMs created after the start   :", (meta["timestamp_vm_created"] > 0).sum())
print("VMs deleted before last reading:", (meta["timestamp_vm_deleted"] <= cpu["timestamp"].max()).sum())

**Found:** a flat line, exactly 10,000 VMs at all times. No VM is created or deleted.
**Why:** the dataset only keeps VMs that lived through the whole window.
**Decision:** forecast the count with persistence (error = 0) and spend the effort on CPU.
**Limitation:** real clouds have VMs starting and stopping, and this dataset hides that.

## 4. What does average CPU look like over time?
**Question:** Is it noisy, trending or smooth?

In [ ]:
by_time = tr.groupby("timestamp")["avg_cpu"]
mean_cpu = by_time.mean()
median_cpu = by_time.median()
p95_cpu = by_time.quantile(0.95)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
sns.lineplot(x=hours, y=mean_cpu.values, ax=axes[0]);   axes[0].set_title("Mean")
sns.lineplot(x=hours, y=median_cpu.values, ax=axes[1]); axes[1].set_title("Median")
sns.lineplot(x=hours, y=p95_cpu.values, ax=axes[2]);    axes[2].set_title("95th percentile")
for ax in axes:
    ax.set_xlabel("hours")
plt.tight_layout()
plt.savefig("../reports/figures/04_aggregate.png", dpi=150, bbox_inches="tight")
plt.show()

step = mean_cpu.diff().dropna()          # change from one step to the next
print("average level      :", round(mean_cpu.mean(), 2))
print("lowest / highest   :", round(mean_cpu.min(), 2), "/", round(mean_cpu.max(), 2))
print("typical 5-min jump :", round(step.std(), 3))

**Found:** the mean stays near 9.4% and moves only about ±0.2 points.
**Why:** averaging 10,000 VMs cancels most random ups and downs.
**Decision:** easy to forecast, so a simple model may be enough. But a calm average can hide risk (see section 8).

## 5. Is there a pattern in time?
**Question:** Trend? Repeating cycle? Pull back to the average?
**What I did:** autocorrelation (ACF) = how similar the series is to itself *k* steps earlier. Computed with pandas, plotted as bars.

In [ ]:
lags = range(1, 31)
acf = [mean_cpu.autocorr(k) for k in lags]       # similarity with k steps earlier

plt.figure(figsize=(8, 3))
sns.barplot(x=list(lags), y=acf, color="steelblue")
band = 1.96 / np.sqrt(len(mean_cpu))             # values inside +/- band are not clearly different from 0
plt.axhline(band, color="red", linestyle="--")
plt.axhline(-band, color="red", linestyle="--")
plt.title("Autocorrelation of the mean CPU")
plt.xlabel("lag (5-minute steps)"); plt.ylabel("autocorrelation")
plt.savefig("../reports/figures/05_acf.png", dpi=150, bbox_inches="tight")
plt.show()

half = len(mean_cpu) // 2
print("average of first half :", round(mean_cpu.iloc[:half].mean(), 2))
print("average of second half:", round(mean_cpu.iloc[half:].mean(), 2))
print("ACF at lag 1, 6, 12   :", [round(mean_cpu.autocorr(k), 2) for k in (1, 6, 12)])

**Found:** ACF is about 0.68 at lag 1 and fades slowly (0.53 at lag 6, 0.44 at lag 12). The two halves have similar averages, so no clear trend.
**Why:** the series wanders a little but keeps returning to its average (mean reversion).
**Decision:**
- try models that pull back to the average (AR, ARMA);
- persistence should be weak, because the last value is noisy;
- a daily cycle needs 24 hours and I only have about 11, so I cannot test it.

## 6. What does CPU of a single VM look like?
**Question:** Is load spread across many VMs, or caused by a few very busy ones?

In [ ]:
vm_average = tr.groupby("vm_id")["avg_cpu"].mean()      # one average per VM

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
sns.histplot(tr["avg_cpu"], bins=100, ax=axes[0]);  axes[0].set_title("All readings")
sns.histplot(vm_average, bins=100, ax=axes[1]);     axes[1].set_title("Average per VM")
for ax in axes:
    ax.set_yscale("log")                  # log scale so small bars stay visible
    ax.set_xlabel("avg_cpu %")
plt.tight_layout()
plt.savefig("../reports/figures/06_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

busiest = vm_average.sort_values(ascending=False).head(len(vm_average) // 100)    # busiest 1% of VMs
print("mean   :", round(tr["avg_cpu"].mean(), 2))
print("median :", round(tr["avg_cpu"].median(), 2))
print("share of all CPU used by the busiest 1% of VMs:", round(busiest.sum() / vm_average.sum() * 100, 1), "%")

**Found:** most VMs are quiet (median about 5%), a few are busy (mean about 9.4%). The busiest 1% of VMs use only about 10% of all CPU.
**Why:** load is spread broadly. My first guess, "a few hot VMs drive the mean", was wrong, and I corrected it with the data.
**Decision:** a fleet-level forecast makes sense.

## 7. What happens in busy and quiet periods?
**Question:** When the average is high, do many VMs rise a little, or a few rise a lot?

In [ ]:
peak_times = mean_cpu[mean_cpu >= mean_cpu.quantile(0.9)].index    # top 10% timestamps
low_times = mean_cpu[mean_cpu <= mean_cpu.quantile(0.1)].index     # bottom 10% timestamps

peak_by_vm = tr[tr["timestamp"].isin(peak_times)].groupby("vm_id")["avg_cpu"].mean()
low_by_vm = tr[tr["timestamp"].isin(low_times)].groupby("vm_id")["avg_cpu"].mean()
difference = peak_by_vm - low_by_vm            # how much each VM rises in a peak

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
sns.lineplot(x=hours, y=mean_cpu.values, ax=axes[0])
sns.scatterplot(x=(peak_times - train_times[0]) / 3600, y=mean_cpu[peak_times].values, color="red", label="top 10%", ax=axes[0])
sns.scatterplot(x=(low_times - train_times[0]) / 3600, y=mean_cpu[low_times].values, color="green", label="bottom 10%", ax=axes[0])
axes[0].set_title("Busy and quiet periods"); axes[0].set_xlabel("hours")
sns.histplot(difference.clip(-5, 5), bins=80, ax=axes[1])
axes[1].axvline(0, color="black")
axes[1].set_title("Change per VM (peak minus low)"); axes[1].set_xlabel("points (cut at ±5)")
plt.tight_layout()
plt.savefig("../reports/figures/07_peaks.png", dpi=150, bbox_inches="tight")
plt.show()

print("gap between peak and low average:", round(mean_cpu[peak_times].mean() - mean_cpu[low_times].mean(), 2))
print("share of VMs that are higher in peaks:", round((difference > 0).mean() * 100, 1), "%")
biggest = difference.abs().sort_values(ascending=False).head(100).index      # the 100 biggest movers
print("share of the total change from the 100 biggest movers:", round(difference[biggest].sum() / difference.sum() * 100, 1), "%")

**Found:** *(write this after you see the output)*. If about half of VMs go up and the 100 biggest movers explain a small share, peaks are a small rise on many VMs. If the biggest movers explain most of it, a few VMs cause the peaks.
**Decision:** many small rises → a percentage safety margin for the whole fleet. A few big movers → alerts for single VMs.

## 8. Where is the real risk?
**Question:** The average is calm. Are there VMs that run hot?
**What I did:** counted VMs above 50% and 80%, and compared `max_cpu` (highest point inside 5 minutes) with `avg_cpu`.

In [ ]:
tr["above_50"] = tr["avg_cpu"] > 50
tr["above_80"] = tr["avg_cpu"] > 80
share = tr.groupby("timestamp")[["above_50", "above_80"]].mean() * 100     # % of VMs per timestamp
hot_counts = tr[tr["above_80"]].groupby("vm_id").size()                    # intervals each hot VM spent above 80%

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
sns.lineplot(x=hours, y=share["above_50"].values, label="above 50%", ax=axes[0])
sns.lineplot(x=hours, y=share["above_80"].values, label="above 80%", ax=axes[0])
axes[0].set_title("% of VMs that are hot"); axes[0].set_xlabel("hours")
sns.histplot(tr["max_cpu"] - tr["avg_cpu"], bins=100, ax=axes[1])
axes[1].set_yscale("log"); axes[1].set_title("max_cpu minus avg_cpu"); axes[1].set_xlabel("points")
sns.histplot(hot_counts, bins=30, ax=axes[2])
axes[2].set_yscale("log"); axes[2].set_title("Intervals above 80% per hot VM"); axes[2].set_xlabel("intervals")
plt.tight_layout()
plt.savefig("../reports/figures/08_risk.png", dpi=150, bbox_inches="tight")
plt.show()

print("average % of VMs above 50%:", round(share["above_50"].mean(), 2))
print("average % of VMs above 80%:", round(share["above_80"].mean(), 2))
print("VMs above 80% at least once:", len(hot_counts))
print("VMs above 80% in more than half of the intervals:", (hot_counts > len(train_times) / 2).sum())
print("average max_cpu:", round(tr["max_cpu"].mean(), 2), "| average avg_cpu:", round(tr["avg_cpu"].mean(), 2))

**Found:** *(write this after you see the output)*. How many VMs are hot? Are the same VMs hot all the time, or does it change? How much higher is `max_cpu` than `avg_cpu`?
**Why:** `avg_cpu` hides short spikes inside each 5-minute interval.
**Decision:** for capacity planning I will not trust the mean alone. I will add a safety margin and look at tail measures (share above 80%, 95th percentile).

## 9. Do VM groups behave differently?
**Question:** Do category, cores or memory matter?

In [ ]:
by_category = tr.groupby(["timestamp", "vm_category"])["avg_cpu"].mean().unstack()
by_category.index = hours                      # use hours on the x-axis

plt.figure(figsize=(8, 3))
sns.lineplot(data=by_category)
plt.title("Average CPU by VM category")
plt.xlabel("hours"); plt.ylabel("%")
plt.savefig("../reports/figures/09_category.png", dpi=150, bbox_inches="tight")
plt.show()

print(meta["vm_category"].value_counts())
print(by_category.mean().round(2))
for column in ["virtual_core_bucket", "memory_gb_bucket"]:
    table = tr.groupby(column)["avg_cpu"].mean().round(2).to_frame("mean_cpu")
    table["n_vms"] = meta.groupby(column).size()
    print(table)

**Found:** Interactive VMs run about 3 points higher than Delay-insensitive ones. The Unknown group wiggles more only because it has few VMs (729). The buckets `>24` cores and `>64` GB have only 10 VMs each, so I read nothing into them.
**Decision:** group labels never change, so they are safe to use (no leakage). I would only use them if validation shows a real gain.

## 10. What could leak future information?

| Column | Problem | Used in models? |
|---|---|---|
| `avg_cpu`, `max_cpu`, `p95_max_cpu` in the metadata file | calculated over the **whole** VM lifetime, including the holdout | **No** |
| `timestamp_vm_deleted` | reveals the future (when a VM ends) | **No** (only to describe the data) |
| `timestamp_vm_created` | always 0, no information | No |
| `vm_category`, cores, memory | fixed labels, known in advance | Safe |
| past CPU values from the training window | known at forecast time | Safe |

**Decision:** models only see past values of the target series.

## 11. Summary

| Found | Evidence | Decision |
|---|---|---|
| Data is clean | no gaps, duplicates or missing values | no cleaning |
| VM count is constant | 10,000 at every time | persistence for the count |
| Average CPU is smooth | tiny 5-minute changes | simple models can work |
| Returns to its average | ACF 0.68 / 0.53 / 0.44, no trend | try AR/ARMA, ETS, ridge |
| Only 11 hours | no daily cycle visible | seasonal model is an experiment |
| Load is spread widely | busiest 1% use about 10% of CPU | fleet-level forecast |
| Average hides spikes | `max_cpu` far above `avg_cpu` | safety margin and tail measures |
| Some metadata leaks the future | covers the holdout period | excluded |

**Next:** compare models with time-aware validation, then build the capacity-planning rule.